In [ ]:
#Application Programming Interface (API)
"""
COMPREHENSIVE FLASK-MONGODB REST API
This application demonstrates:
1. MongoDB connection and CRUD operations
2. RESTful endpoints with proper HTTP methods
3. Error handling and validation
4. Request parsing and response formatting
"""

from flask import Flask, request, jsonify
from pymongo import MongoClient
from bson.objectid import ObjectId
from bson.json_util import dumps
import os
from dotenv import load_dotenv
from werkzeug.exceptions import HTTPException

# Load environment variables
load_dotenv()

# Initialize Flask app
app = Flask(__name__)

# MongoDB connection setup
MONGODB_URI = os.getenv('MONGODB_URI', 'mongodb://localhost:27017/')
client = MongoClient(MONGODB_URI)
db = client['library_api']  # Database name
books_collection = db['books']  # Collection name

# Error handling
@app.errorhandler(HTTPException)
def handle_exception(e):
    """Return JSON instead of HTML for HTTP errors."""
    response = e.get_response()
    response.data = dumps({
        "error": {
            "code": e.code,
            "name": e.name,
            "description": e.description,
        }
    })
    response.content_type = "application/json"
    return response

# Helper functions
def validate_book_data(data, partial_update=False):
    """Validate book data for creation or update"""
    if not isinstance(data, dict):
        return {"valid": False, "error": "Data must be a JSON object"}
    
    required_fields = ['title', 'author', 'year'] if not partial_update else []
    for field in required_fields:
        if field not in data:
            return {"valid": False, "error": f"Missing required field: {field}"}
    
    if 'year' in data and not isinstance(data['year'], int):
        return {"valid": False, "error": "Year must be an integer"}
    
    return {"valid": True}

# API Routes
@app.route('/')
def home():
    """API Homepage with documentation"""
    return """
    <h1>Library API</h1>
    <p>Available endpoints:</p>
    <ul>
        <li>GET /books - List all books</li>
        <li>POST /books - Add a new book</li>
        <li>GET /books/&lt;id&gt; - Get a specific book</li>
        <li>PUT /books/&lt;id&gt; - Update a book</li>
        <li>DELETE /books/&lt;id&gt; - Delete a book</li>
    </ul>
    """

@app.route('/books', methods=['GET'])
def get_books():
    """
    GET all books
    ---
    tags:
      - Books
    responses:
      200:
        description: List of all books
    """
    books = list(books_collection.find({}))
    return jsonify({"books": books}), 200

@app.route('/books', methods=['POST'])
def add_book():
    """
    POST create new book
    ---
    tags:
      - Books
    parameters:
      - in: body
        name: body
        required: true
        schema:
          type: object
          properties:
            title:
              type: string
            author:
              type: string
            year:
              type: integer
    responses:
      201:
        description: Book created
      400:
        description: Invalid input
    """
    if not request.is_json:
        return jsonify({"error": "Request must be JSON"}), 400
    
    data = request.get_json()
    validation = validate_book_data(data)
    if not validation['valid']:
        return jsonify({"error": validation['error']}), 400
    
    # Insert new book
    result = books_collection.insert_one(data)
    new_book = books_collection.find_one({"_id": result.inserted_id})
    
    return jsonify({"book": new_book}), 201

@app.route('/books/<string:book_id>', methods=['GET'])
def get_book(book_id):
    """
    GET single book by ID
    ---
    tags:
      - Books
    parameters:
      - name: book_id
        in: path
        required: true
        type: string
    responses:
      200:
        description: Book found
      404:
        description: Book not found
    """
    try:
        book = books_collection.find_one({"_id": ObjectId(book_id)})
    except:
        return jsonify({"error": "Invalid book ID format"}), 400
    
    if book:
        return jsonify(book), 200
    return jsonify({"error": "Book not found"}), 404

@app.route('/books/<string:book_id>', methods=['PUT'])
def update_book(book_id):
    """
    PUT update entire book
    ---
    tags:
      - Books
    parameters:
      - name: book_id
        in: path
        required: true
        type: string
      - in: body
        name: body
        required: true
        schema:
          type: object
          properties:
            title:
              type: string
            author:
              type: string
            year:
              type: integer
    responses:
      200:
        description: Book updated
      400:
        description: Invalid input
      404:
        description: Book not found
    """
    if not request.is_json:
        return jsonify({"error": "Request must be JSON"}), 400
    
    data = request.get_json()
    validation = validate_book_data(data)
    if not validation['valid']:
        return jsonify({"error": validation['error']}), 400
    
    try:
        result = books_collection.update_one(
            {"_id": ObjectId(book_id)},
            {"$set": data}
        )
    except:
        return jsonify({"error": "Invalid book ID format"}), 400
    
    if result.matched_count == 0:
        return jsonify({"error": "Book not found"}), 404
    
    updated_book = books_collection.find_one({"_id": ObjectId(book_id)})
    return jsonify(updated_book), 200

@app.route('/books/<string:book_id>', methods=['DELETE'])
def delete_book(book_id):
    """
    DELETE remove book
    ---
    tags:
      - Books
    parameters:
      - name: book_id
        in: path
        required: true
        type: string
    responses:
      200:
        description: Book deleted
      404:
        description: Book not found
    """
    try:
        result = books_collection.delete_one({"_id": ObjectId(book_id)})
    except:
        return jsonify({"error": "Invalid book ID format"}), 400
    
    if result.deleted_count == 0:
        return jsonify({"error": "Book not found"}), 404
    
    return jsonify({"message": "Book deleted successfully"}), 200

if __name__ == '__main__':
    # Create some sample data if collection is empty
    if books_collection.count_documents({}) == 0:
        sample_books = [
            {"title": "Python Crash Course", "author": "Eric Matthes", "year": 2019},
            {"title": "Fluent Python", "author": "Luciano Ramalho", "year": 2015},
            {"title": "Deep Learning", "author": "Ian Goodfellow", "year": 2016}
        ]
        books_collection.insert_many(sample_books)
    
    # Run the application
    port = int(os.environ.get("PORT", 5000))
    app.run(host='0.0.0.0', port=port, debug=True)